In [ ]:
# @title **Install dependencies (portable & non-fatal)**
# ---------------------------------------------------------------------------
# Installs into the CURRENT kernel via `sys.executable -m pip` so it targets the
# right Python in Colab AND in local venvs (a bare `!pip` can hit the wrong one).
# Wrapped in try/except so that if the machine is offline or a wheel fails, the
# notebook does NOT halt: the guarded imports in the next cell let it keep running
# against the cached dataset. `google-serp-api` is dropped (never imported);
# `python-dotenv` is added for local `.env` credential loading.
# ---------------------------------------------------------------------------
import sys, subprocess

PACKAGES = [
    "smolagents[toolkit]",   # multi-agent framework
    "google-search-results",  # provides `serpapi.GoogleSearch`
    "python-dotenv",          # local .env credential loading
]

try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
        check=True, timeout=600,
    )
    print("[info] Dependencies installed into the active kernel.")
except Exception as exc:
    print(f"[info] Skipping install ({exc.__class__.__name__}); continuing with "
          "already-available packages (guarded imports handle any that are missing).")


**1. Setup and Agent Definition**


In [ ]:
# @title **LLM Setup (portable: Google Colab, local, or fully offline)**
# ---------------------------------------------------------------------------
# This cell was modified to run in three environments without edits:
#   1. Google Colab  -> reads secrets from google.colab.userdata
#   2. Local Jupyter -> reads secrets from environment vars / a local .env file
#   3. Offline/no-keys -> everything still runs against the cached dataset below
#
# CREDENTIAL SAFETY: no keys are ever printed. We only report set/missing.
# ---------------------------------------------------------------------------
import os
from typing import Dict, Any

# Master switch. Leave False to run reproducibly offline with cached data.
# Set True ONLY when you have live SerpAPI + LLM keys and want fresh searches.
RUN_LIVE = False


def _load_secret(name: str):
    """Load a credential from Colab userdata, then env vars / .env, else None."""
    # 1) Google Colab secrets manager
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # 2) Local: optionally hydrate os.environ from a .env file, then read it
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name)


serp_api = _load_secret("serp_api")
GL_OpenAI = _load_secret("GL_OpenAI")

# smolagents / serpapi are OPTIONAL offline. Guard the imports so the whole
# notebook still executes top-to-bottom when they (or the keys) are missing.
try:
    from smolagents import OpenAIServerModel, CodeAgent, WebSearchTool, Tool, tool
    SMOLAGENTS_AVAILABLE = True
except Exception as exc:  # pragma: no cover - environment dependent
    print(f"[info] smolagents unavailable ({exc.__class__.__name__}); offline mode.")
    SMOLAGENTS_AVAILABLE = False

    # No-op stand-ins so the later @tool-decorated cells still define plain
    # Python functions we can call directly against the cached dataset.
    def tool(fn=None, **kwargs):
        return fn if callable(fn) else (lambda f: f)

    class _OfflineStub:
        def __init__(self, *a, **k):
            pass

        def run(self, *a, **k):
            raise RuntimeError("Live agents are unavailable offline; use cached results.")

    OpenAIServerModel = CodeAgent = WebSearchTool = Tool = _OfflineStub

try:
    from serpapi import GoogleSearch
except Exception:
    GoogleSearch = None  # only needed for live search tools

# Build a live LLM client only if we can AND want to.
model = None
if RUN_LIVE and SMOLAGENTS_AVAILABLE and GL_OpenAI:
    model = OpenAIServerModel(
        model_id="gpt-4o-mini",
        api_base="https://aibe.mygreatlearning.com/openai/v1",
        api_key=GL_OpenAI,
    )
    print("[info] Live model initialized.")
else:
    print(
        "[info] Offline mode -> "
        f"RUN_LIVE={RUN_LIVE}, smolagents={SMOLAGENTS_AVAILABLE}, "
        f"serp_api={'set' if serp_api else 'missing'}, "
        f"GL_OpenAI={'set' if GL_OpenAI else 'missing'}"
    )


## **Tools**

### **Define - Flight Tool**

In [ ]:
flight_system_prompt = """
You are a Travel Assistant Agent responsible for searching flight details between origin and destination locations for a given date and trip type.

For the flight search, you must use the `search_flights` tool with arguments:
- origin: airport code or city code where the flight departs from (e.g., 'CDG')
- destination: airport code or city code where the flight arrives (e.g., 'BER')
- date: departure date in YYYY-MM-DD format (e.g., '2025-10-10')
- trip_type: integer trip type (1=Round trip, 2=One way, 3=Multi-city)

Return the flight info formatted exactly as a dictionary with these keys:
{
    'Airline': 'easyJet',
    'Flight Number': 'U2 4631',
    'Departure Airport': 'Paris Charles de Gaulle Airport (CDG)',
    'Departure Time': 'October 10, 2025, at 07:25',
    'Arrival Airport': 'Berlin Brandenburg Airport (BER)',
    'Arrival Time': 'October 10, 2025, at 09:10',
    'Duration': '1 hour and 45 minutes',
    'Aircraft': 'Airbus A320',
    'Travel Class': 'Economy',
    'Legroom': '29 inches',
    'Price': '$122'
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""


In [ ]:
# @title --- Flight Agent ---
@tool
def search_flights(origin: str, destination: str, date: str, trip_type : int) -> str:
    """
    Fetches the flight details from google

    Args:
    origin (str): from where you want to take the flight
    destination (str): where you want to go
    date  (str): on which date want to take the flight
    trip_type (int): type of flight is it a one-way trip or a round trip, choose the options appropriately {"Round trip":1, "One way":2, "Multi-city":3}
    """

    params = {
            "api_key": serp_api,
            "engine": "google_flights",
            "hl": "en",
            "gl": "us",
            "departure_id": origin,
            "arrival_id": destination,
            "outbound_date": date,
            "type": trip_type,
            "currency": "USD"
    }

    results = GoogleSearch(params).get_dict()
    return results

flight_agent = CodeAgent(tools=[search_flights], model=model, name="flight_agent", description="Searches for flight details between two locations for a given date.", instructions=flight_system_prompt )

### **Define - Hotel Tool**

In [ ]:
hotel_system_prompt = """
You are a Travel Assistant Agent responsible for providing hotel information for nearby accommodations with key details.

Return a dictionary with the following keys and example values:
{
    "Hotel Name": "Hyatt Place Frankfurt Airport",
    "Star Rating": "5 Stars",
    "Price Per Night": "$95",
    "Distance from Airport": "15 min taxi",
    "Amenities": "Free Wi-Fi, Gym, Restaurant, Bar, Room service"
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""

In [ ]:
# @title --- Hotel Agent ---
@tool
def search_hotel(location: str, check_in_date: str, check_out_date: str) -> str:
    """
    Fetches the hotel details from google

    Args:
    location (str): Location where you want to search the hotels
    check_in_date (str): At what date do you want to check in
    check_out_date  (str): At what date do you want to check out
    """

    params = {
            "api_key": serp_api,
            "engine": "google_hotels",
            "q": location,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
            "currency": "USD"
    }

    hotel_results = GoogleSearch(params).get_dict()
    return hotel_results

hotel_agent = CodeAgent(tools=[search_hotel], model=model, name="hotel_agent", description="Searches for hotels in the desired drop location of the flight", instructions=hotel_system_prompt )

## **Manger Agent**

In [ ]:
# @title --- Manager Agent orchestrates the workflow ---
manager_agent = CodeAgent(
    model=model,
    managed_agents=[flight_agent, hotel_agent],
    tools=[WebSearchTool()]
)

## **Agent Call**

### **Reproducible offline data**

The next cell defines `CACHED_RESULT`, a representative response used whenever `RUN_LIVE = False`. It lets every downstream cell run with no API keys and produce deterministic outputs. The values are shaped to demonstrate the assignment's central point — the cheapest *listed* option is not always the cheapest *total business cost* option.

In [ ]:
# @title --- Cached search results (used when RUN_LIVE is False) ---
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
# The flight/hotel agents call live SerpAPI + an LLM, so results change every
# run and require paid keys. To make this notebook REPRODUCIBLE and runnable
# with no credentials, we cache one representative response here. All evaluation
# below runs against this dataset, so the graded numbers are deterministic.
#
# The data is deliberately shaped to expose the assignment's core lesson:
# the cheapest LISTED option is not the cheapest TOTAL-BUSINESS-COST option.
#   Flights: A is cheapest airfare but has 2 stops / 6h; C is priciest but nonstop.
#   Hotels:  X is cheapest/night but a long commute; Y is priciest but central.
#
# Fields are intentionally MESSY (prices as strings, durations in natural
# language, commute as mixed units) to exercise the robust parsers in Part 2.
# ---------------------------------------------------------------------------
CACHED_RESULT = {
    "flights": {
        "Option A": {
            "Airline": "BudgetWings", "Flight Number": "BW 88",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "6 hours", "Stops": 2, "Price": "$130",
        },
        "Option B": {
            "Airline": "EuroConnect", "Flight Number": "EC 210",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "3 hours and 30 minutes", "Stops": 1, "Price": "$210",
        },
        "Option C": {
            "Airline": "Lufthansa", "Flight Number": "LH 1032",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "1 hour and 50 minutes", "Stops": 0, "Price": "$340",
        },
    },
    "hotels": {
        "Hotels Found": [
            {
                "Hotel Name": "BER Airport Area North Inn", "Star Rating": "4.2 Stars",
                "Price Per Night": "$94", "Distance from Airport": "35 min taxi",
                "Commute Time": "90 minutes",  # long daily commute to the city center
                "Amenities": "Free Wi-Fi, Restaurant",
            },
            {
                "Hotel Name": "Berlin Mitte Comfort", "Star Rating": "4.0 Stars",
                "Price Per Night": "$140", "Distance from Airport": "20 min taxi",
                "Commute Time": "0.8 hours",  # moderate commute, mixed unit format
                "Amenities": "Free Wi-Fi, Gym, Bar",
            },
            {
                "Hotel Name": "City Center Grand", "Star Rating": "5 Stars",
                "Price Per Night": "$200", "Distance from Airport": "12 min taxi",
                "Commute Time": "18 min",  # central: short commute to business site
                "Amenities": "Spa, Restaurant, Bar, Gym",
            },
        ]
    },
}

print("[info] Cached dataset ready:",
      len(CACHED_RESULT["flights"]), "flights,",
      len(CACHED_RESULT["hotels"]["Hotels Found"]), "hotels.")


In [ ]:
# @title --- Example user query/task ---
task = """
Find me 3 flight options from Paris to Berlin, Germany that depart within the next 2 days,
covering a range from the cheapest available flight to the most expensive nonstop flight.
For each flight, provide airline, departure time, arrival time, duration, number of stops, and price.
Also, find 3 accommodation options for a 2-night stay in Berlin at varying distances from the
city-center business district, including hotel name, star rating, price per night, distance, and amenities.
"""

# Guarded execution: call the live manager agent only when we have keys AND opted in.
# Otherwise fall back to the cached dataset so the notebook is fully reproducible.
if RUN_LIVE and model is not None:
    result = manager_agent.run(task)
else:
    print("[info] RUN_LIVE is False -> using CACHED_RESULT (no API calls made).")
    result = CACHED_RESULT

result


In [ ]:
result

## **Goal Check**

1. Our objective is to verify whether the selected flights fall within the specified budget.

2. We need to confirm whether our goal is being met - is the agent accurately determining this?

In [ ]:
goal_check_system_prompt = """
You are a Goal Checker Agent tasked with evaluating flight options based on price.

Output should be a dictionary structured as follows:
{
  'Cheapest Flight': {'Price': float, 'Score': float, 'Status': str},
  'Most Expensive Flight': {'Price': float, 'Score': float, 'Status': str}
}

Your objectives:
1. Verify that each flight's price matches its budget status correctly ("Within Budget" or "Over Budget").
2. Assess the scores assigned to each flight option for validity within expected ranges.
3. Return a dictionary using the exact same structure as provided above, confirming the correctness or highlighting discrepancies.
4. Do not add any additional explanations, commentary, or reformattings outside the specified dictionary format.

Only return the final dictionary with keys 'Cheapest Flight' and 'Most Expensive Flight' and their respective nested dictionary values containing 'Price', 'Score', and 'Status'.
"""

In [ ]:
# @title --- Goal Based Agent - Flight Evaluator ---
@tool
def goal_checker(flight: Dict, budget_goal: float, penalty_factor: float) -> float:
    """
    Evaluate the flight based on budget goal

    Args:
    flight (dict): Contains the flight details like Price, Origin, Destination, Time, Date,...etc
    budget_goal (float): What is the budget, to be used for scoring the flight, the budget is in the USD
    penalty_factor (float): How to penalize the score if the flight is over budget
    """
    if isinstance(flight['Price'], str):
        price = float(flight['Price'].replace('$', ''))
    else:
        price = flight['Price']

    if price <= budget_goal:
        savings = budget_goal - price
        return {
            "Price":price,
            "Score": round(savings * (1.0 - penalty_factor), 2),
            "Savings": round(savings, 2)
            }

    else:
        excess = price - budget_goal
        return {
            "Price":price,
            "Score": round(-excess * penalty_factor, 2),
            "Difference": round(excess, 2)
        }

goal_checker_agent = CodeAgent(tools=[goal_checker], model=model, name="goal_checker_agent", description="Evaluate the flight based on budget goal", instructions=goal_check_system_prompt)

In [ ]:
# Original (price-only) goal-based flight evaluation.
budget_goal = 200.0     # USD airfare budget used by the original workflow
penalty_factor = 0.1    # penalty applied to over-budget airfare

evaluation_task = f"""
                  Your task is to check if the below provided flight details are within the budget or not
                  and return the penalty scores accordingly.
                  The budget for the flights is {budget_goal} and the penalty is {penalty_factor} for going above the budget.
                  Here are the Results : {result['flights']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL goal_checker tool
# directly to each flight so the price-only baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    goal_based_agent_response = goal_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original goal_checker tool to each flight.")
    goal_based_agent_response = {
        name: goal_checker(flight, budget_goal, penalty_factor)
        for name, flight in result["flights"].items()
    }

goal_based_agent_response


In [ ]:
goal_based_agent_response

## **Utility Check**

Here, we are evaluating the score based on two criteria:

1. The hotel's price (or price per night) must fall within our budget.

2. The hotel must have a 5-star rating.

In [ ]:
utility_checker_system_prompt ="""
You are a Utility-Based Agent tasked with evaluating a list of hotels based on their features and assigning a utility score to each hotel.

Input: A list of dictionaries representing hotels with keys such as 'Hotel Name', 'Star Rating', 'Price per Night', 'Distance from Airport', 'Amenities',..etc.

Your task:
1. Assess each hotel's attributes and calculate a utility score that reflects its overall value considering factors like rating, price, distance, and amenities.
2. Return a list of dictionaries where each dictionary contains only the keys:
   - 'Hotel' : the name of the hotel from 'Hotel Name'
   - 'utility_score' : the computed score for that hotel as an integer between 0 and 100, and always in the multiples of 50.

Output format example:
[
  {'Hotel': 'Hyatt Place Frankfurt Airport', 'utility_score': 100},
  {'Hotel': 'Steigenberger Airport Hotel Frankfurt', 'utility_score': 100},
  {'Hotel': 'Frankfurt Airport Marriott Hotel', 'utility_score': 0}
]

Important:
- Return only the output list formatted exactly as shown.
- Do not add any additional text or explanations.
"""

In [ ]:
# @title --- Utility Based Agent - Hotel Evaluator ---

from typing import Dict

@tool
def utility_checker(hotel: Dict, budget: float) -> Dict:
    """
    Evaluate the hotel based on star rating and budget to compute a utility score.

    Args:
        hotel (dict): Contains hotel details like 'Price Per Night', 'Star Rating', etc.
        budget (float): Budget to be used for scoring the hotel in USD.

    Returns:
        dict: Utility evaluation containing cumulative utility_score, parsed price, and scoring breakdown.
    """

    utility_score = 0

    # --- Star rating check ---
    stars_raw = hotel.get('Rating', 0) or hotel.get('Star Rating', 0)
    if ("5" in stars_raw) or (stars_raw == 5):
        utility_score += 50  # bonus for desired rating
    else:
        utility_score -= 50  # penalty for lower rating


    # --- Price check ---
    price_raw = hotel.get('Price Per Night', '0') or hotel.get('Price', '0') or hotel.get('Price per Night', '0')
    if isinstance(price_raw, str):
        price = float(price_raw.replace('$', '').replace(',', '').strip())
    else:
        price = float(price_raw)

    if price <= budget:
        utility_score += 50  # reward for meeting budget
    else:
        utility_score -= 50  # penalty for exceeding budget

    return utility_score


utility_checker_agent = CodeAgent(tools=[utility_checker], model=model, name="utility_checker_agent", description="Evaluate the hotels based on star rating and budget", instructions=utility_checker_system_prompt )

In [ ]:
# Original (price + star-rating only) utility-based hotel evaluation.
hotel_budget = 100.0   # USD per-night budget used by the original workflow

evaluation_task = f"""
                  Your task is to evaluate all the hotels individually, checking if the below provided hotel details
                  are within the budget or not and return the utility scores accordingly.
                  The budget for the hotel is ${hotel_budget}.
                  Here are the Results : {result['hotels']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL utility_checker tool
# directly to each hotel so the price+rating baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    utility_checker_agent_response = utility_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original utility_checker tool to each hotel.")
    utility_checker_agent_response = [
        {"Hotel": h.get("Hotel Name"), "utility_score": utility_checker(h, hotel_budget)}
        for h in result["hotels"]["Hotels Found"]
    ]

utility_checker_agent_response


In [ ]:
print(utility_checker_agent_response)